In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(".."))
from src.config import *
from src.io import *
from src.procesamiento import *
from src.visualizacion import *
from src.funciones_complejas import *

# Imports y configuraciones basicas
import os
import ast
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns

# ==========================================
# CONFIGURACIONES GLOBALES
# ==========================================
sns.set_style("whitegrid") 
# from src.config import crear_directorios_overleaf
# crear_directorios_overleaf()  # Crea subcarpetas en graficos_overleaf/

In [4]:
# CARGA Y LIMPIEZA DE DATOS (Nomenclatura completa)
# ==========================================
df_base = pd.read_excel("../data/pacientes.xlsx")
hospitales = pd.read_csv("../data/hospitales_coordenadas.csv")

# Diccionarios de referencia para hospitales
dict_complejidad = dict(zip(hospitales['Nombre Hospital'], hospitales['complejidad']))
hospitales['color_rgb'] = hospitales['color'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# --- RECUPERADO: El renombre COMPLETO de columnas ---
df_base = df_base.rename(columns={
    'Id Hospital': 'hospital_id',
    'Nombre Hospital': 'hospital_origen',
    'Id': 'paciente_id',
    'Fecha inicio': 'fecha_ingreso',
    'Estado al ingreso': 'estado_ingreso',
    'Tipo al ingreso': 'tipo_ingreso',
    'Último estado': 'estado_ultimo',
    'Último tipo': 'tipo_ultimo',
    'Sexo': 'sexo',
    'Edad': 'edad',
    'Nivel riesgo clínico': 'riesgo_clinico',
    'Nivel riesgo social': 'riesgo_social',
    'Enfermedades preexistentes Covid-19': 'comorbilidades_covid',
    'Enfermedades preexistentes pediatría': 'comorbilidades_pediatria',
    'Vacuna': 'vacuna',
    'Cant. dosis': 'cantidad_dosis',
    '1º dosis': 'fecha_dosis_1',
    '2º dosis': 'fecha_dosis_2',
    'Buscado en el ministerio': 'validado_ministerio',
    'Obra social': 'obra_social',
    'Asistencia Respiratoria Mecánica': 'requiere_arm',
    'Motivo': 'motivo_egreso',
    'Operación': 'operacion',
    'Fecha egreso': 'fecha_egreso',
    'Última actualización': 'fecha_ultima_actualizacion',
    'Pasó por Críticas': 'paso_criticas',
    'Pasó por Intermedias': 'paso_intermedias',
    'Pasó por Generales': 'paso_generales'
}).copy()

df_base['hospital_origen'] = df_base['hospital_origen'].replace({
    'Módulo Hospitalario 11- FV': 'Módulo Hospitalario 11 - FV',
    'Módulo Hospitalario  9 - AB': 'Módulo Hospitalario 9 - AB'
})

df_base['fecha_ingreso'] = pd.to_datetime(df_base['fecha_ingreso'], errors='coerce')
df_base['fecha_egreso'] = pd.to_datetime(df_base['fecha_egreso'], errors='coerce')
    # Normalización proactiva de nombres de hospitales
df_base['hospital_origen'] = df_base['hospital_origen'].str.strip()
# --- NUEVO: Mapeo de IDs únicos de Hospital ---
df_base['Nombre Hospital'] = df_base['hospital_origen'] 
df_base = mapear_ids_hospitales(df_base, hospitales, drop_missing=True)


    # Ordenamiento crítico para el cálculo de traslados funcionales
df_base = df_base.sort_values(['paciente_id', 'fecha_ingreso']).reset_index(drop=True)

# Cálculo de estancia por segmento (hospital) con redondeo ceil
df_base['dias_en_nodo'] = np.ceil((df_base['fecha_egreso'] - df_base['fecha_ingreso']).dt.total_seconds() / 86400).fillna(0).astype('Int64')


# ==========================================
# 2. MÉTRICAS DE TRAYECTORIA DEL PACIENTE (RED)
# ==========================================
# Agrupamos por paciente para extraer el inicio, fin y los motivos
df_metricas_globales = df_base.groupby('paciente_id').agg(
    fecha_ingreso_red=('fecha_ingreso', 'first'),  
    fecha_egreso_red=('fecha_egreso', 'last'),     
    dias_estadia_total=('dias_en_nodo', 'sum'),
    motivos_historial=('motivo_egreso', list),     
    motivo_fin_caso=('motivo_egreso', 'last')      
).reset_index()

# Validamos que el motivo de fin de caso
condiciones = [
    df_metricas_globales['motivo_fin_caso'] == 'alta-domiciliaria',
    df_metricas_globales['motivo_fin_caso'] == 'muerte',
    df_metricas_globales['motivo_fin_caso'] == 'traslado-otro',
    df_metricas_globales['motivo_fin_caso'] == 'traslado-extra-sanitario'
]
resultados = ['alta', 'muerte', 'hospital externo', 'alta hotel']

df_metricas_globales['motivo_fin_caso'] = np.select(condiciones, resultados, default='otro/desconocido')
df_base = df_base.merge(df_metricas_globales, on='paciente_id', how='left')


# ==========================================
# 3. CORE DEL MODELO: CONSTRUCCIÓN DE TRASLADOS (PARA MAPAS)
# ==========================================
df_base['id_hospital_destino'] = df_base.groupby('paciente_id')['id_hospital'].shift(-1)
df_base['hospital_destino'] = df_base.groupby('paciente_id')['hospital_origen'].shift(-1)
df_base['fecha_ingreso_destino'] = df_base.groupby('paciente_id')['fecha_ingreso'].shift(-1)
df_base['estado_destino'] = df_base.groupby('paciente_id')['estado_ingreso'].shift(-1)
df_base['tipo_destino'] = df_base.groupby('paciente_id')['tipo_ingreso'].shift(-1) 

df_base['dias_traslado'] = (df_base['fecha_ingreso_destino'] - df_base['fecha_egreso']).dt.days
df_base.loc[df_base['dias_traslado'] == -1, 'dias_traslado'] = 0

    # CRITERIO DE TRASLADO: 
    # 1. El hospital de origen es distinto al de destino.
    # 2. El motivo de egreso contiene la palabra 'traslado'.
mask_traslados = (
    df_base['hospital_destino'].notna() & 
    df_base['motivo_egreso'].str.contains('traslado', na=False, case=False) & 
    (df_base['hospital_origen'] != df_base['hospital_destino'])
)

df_potenciales = df_base[mask_traslados].copy() 
df_aristas_traslados = df_potenciales[df_potenciales['dias_traslado'] <= 100].copy()
df_aristas_traslados = df_aristas_traslados.rename(columns={'hospital_origen': 'hospital_ingreso'})

df_aristas_traslados['alerta_demora'] = df_aristas_traslados['dias_traslado'] > 3
df_aristas_traslados['dias_alerta'] = df_aristas_traslados.apply(lambda row: row['dias_traslado'] if row['alerta_demora'] else 0, axis=1)


# ==========================================
# 4. TABLAS RELACIONALES: ESTANCIAS Y TRAYECTORIAS
# ==========================================
# A. Armamos las Estancias (Episodios) sin perder el último destino
df_estancias_episodios = df_base.sort_values(['paciente_id', 'fecha_ingreso']).copy()
df_estancias_episodios['orden_episodio'] = df_estancias_episodios.groupby('paciente_id').cumcount() + 1
df_estancias_episodios['dias_en_nodo'] = np.ceil((df_estancias_episodios['fecha_egreso'] - df_estancias_episodios['fecha_ingreso']).dt.total_seconds() / 86400).astype('Int64')

# Pegamos la complejidad a cada internación
df_estancias_episodios = df_estancias_episodios.merge(
    hospitales[['Nombre Hospital', 'complejidad']], 
    left_on='hospital_origen', right_on='Nombre Hospital', how='left'
)
df_estancias_episodios['nivel_complejidad'] = df_estancias_episodios['complejidad'].fillna('Desc').astype(str).str.replace('.0', '', regex=False)

# B. Armamos las rutas (Trayectorias)
df_rutas = df_estancias_episodios.groupby('paciente_id').agg(
    ruta_hospitales_str=('hospital_origen', lambda x: ' -> '.join(x.astype(str))),
    ruta_complejidad_str=('nivel_complejidad', lambda x: ' -> '.join(x.astype(str))),
    cantidad_traslados=('hospital_origen', lambda x: len(x) - 1)
).reset_index()

# Unimos con las métricas para tener 1 fila por paciente con su resumen total
df_pacientes_trayectorias = df_metricas_globales.merge(df_rutas, on='paciente_id', how='left')
# Métrica calculada por suma de episodios
# df_pacientes_trayectorias['dias_estadia_total'] = np.ceil((df_pacientes_trayectorias['fecha_egreso_red'] - df_pacientes_trayectorias['fecha_ingreso_red']).dt.total_seconds() / 86400).astype('Int64')
df_pacientes_trayectorias['motivo_fin_caso'] = df_pacientes_trayectorias['motivo_fin_caso'].replace('otro/desconocido', 'alta')


# ==========================================
# 5. EL PEDIDO DE LOS MENTORES: TRAYECTORIAS DE 1 TRASLADO
# ==========================================
# Filtramos solo pacientes con 1 traslado (exactamente 2 pasos/nodos)
df_analisis_rutas = df_pacientes_trayectorias[df_pacientes_trayectorias['cantidad_traslados'] == 1].copy()

# Rescatamos la cantidad de días que pasaron en el "orden_episodio == 1"
tiempo_primer_salto = df_estancias_episodios[df_estancias_episodios['orden_episodio'] == 1][['paciente_id', 'dias_en_nodo']]
tiempo_primer_salto = tiempo_primer_salto.rename(columns={'dias_en_nodo': 'tiempo_hasta_traslado'})

# Unimos la info y armamos el DataFrame final con los nombres declarativos que pidieron
df_tiempos_1_traslado = df_analisis_rutas.merge(tiempo_primer_salto, on='paciente_id', how='left')
df_tiempos_1_traslado = df_tiempos_1_traslado[[
    'ruta_complejidad_str', 
    'motivo_fin_caso', 
    'tiempo_hasta_traslado', 
    'dias_estadia_total'
]].rename(columns={
    'ruta_complejidad_str': 'Tipo de trayectoria',
    'motivo_fin_caso': 'Motivo de egreso',
    'dias_estadia_total': 'Tiempo hasta el egreso (total)'
})


# ==========================================
# 6. PREPARACIÓN DE COORDENADAS (Unificado)
# ==========================================
df_coordenadas = hospitales.rename(columns={'Nombre Hospital': 'hospital', 'Latitud': 'lat', 'Longitud': 'lon'})

# Ajustes manuales unificados
df_coordenadas.loc[df_coordenadas['hospital'] == 'Módulo Hospitalario 8 - LZ', 'lon'] += 0.06
df_coordenadas.loc[df_coordenadas['hospital'] == 'UPA 10 - BE', 'lon'] -= 0.06
df_coordenadas.loc[df_coordenadas['hospital'] == 'Evita Pueblo', 'lon'] -= 0.03

# Desplazar duplicados
nuevas_filas = []
for (lat, lon), group in df_coordenadas.groupby(['lat', 'lon']):
    for i, (_, row) in enumerate(group.iterrows()):
        row_mod = row.copy()
        if i > 0:
            row_mod['lon'] = lon + 0.01   
            row_mod['lat'] = lat + (i * 0.015)  
        nuevas_filas.append(row_mod)

df_coordenadas = pd.DataFrame(nuevas_filas)
hospitales_conocidos = set(df_coordenadas['hospital'])


[AVISO] Ignorando 1 hospitales no registrados:
  - 'MODULO HOSPITALARIO 8 LZ' (61 registros)
Total de registros eliminados: 61


In [5]:
# ---------------------------------------------------------
# PRINT 1: Conocer el terreno (Todas las columnas)
# ---------------------------------------------------------
print("=== LISTA DE COLUMNAS ===")
for col in df_base.columns:
    print(f"- {col}")
print("\n")

=== LISTA DE COLUMNAS ===
- hospital_id
- hospital_origen
- paciente_id
- fecha_ingreso
- estado_ingreso
- tipo_ingreso
- estado_ultimo
- tipo_ultimo
- sexo
- edad
- riesgo_clinico
- riesgo_social
- comorbilidades_covid
- comorbilidades_pediatria
- vacuna
- cantidad_dosis
- fecha_dosis_1
- fecha_dosis_2
- validado_ministerio
- obra_social
- requiere_arm
- motivo_egreso
- operacion
- fecha_egreso
- fecha_ultima_actualizacion
- paso_criticas
- paso_intermedias
- paso_generales
- Nombre Hospital
- id_hospital
- dias_en_nodo
- fecha_ingreso_red
- fecha_egreso_red
- dias_estadia_total
- motivos_historial
- motivo_fin_caso
- id_hospital_destino
- hospital_destino
- fecha_ingreso_destino
- estado_destino
- tipo_destino
- dias_traslado




In [ ]:
# ---------------------------------------------------------
# PRINT 2: Ver los estados de cierre (Motivos de Egreso)
# ---------------------------------------------------------
# Nota: Voy a usar 'Motivo' porque lo vi en el código que me pasaste antes, 
# pero si en el Print 1 ves que se llama distinto, cambialo acá abajo.
print("=== FRECUENCIA DE MOTIVOS DE EGRESO ===")
if 'Motivo' in df_base.columns:
    print(df_base['motivo_egreso'].value_counts(dropna=False))
else:
    print("La columna 'Motivo' no existe. Buscá el nombre correcto en la lista de arriba.")
print("\n")


print("=== FRECUENCIA DE MOTIVOS DE FIN DE CASO ===")
if 'Motivo' in df_base.columns:
    print(df_base['motivo_egreso'].value_counts(dropna=False))
else:
    print("La columna 'Motivo' no existe. Buscá el nombre correcto en la lista de arriba.")
print("\n")



=== FRECUENCIA DE MOTIVOS DE EGRESO ===
La columna 'Motivo' no existe. Buscá el nombre correcto en la lista de arriba.


=== DESTINOS FRECUENTES (si existe la columna) ===


In [ ]:
# ---------------------------------------------------------
# PRINT 3: Ver los destinos (Para tu regla 1)
# ---------------------------------------------------------
print("=== DESTINOS FRECUENTES (si existe la columna) ===")
# Si tenés una columna de destino, reemplazá 'Destino' por el nombre real
if 'Destino' in df_base.columns:
    print(df_base['Destino'].value_counts(dropna=False).head(15))

### ⚠️ OPCIÓN B (Clúster 3): Lógica suelta de ordenamiento
*Nota: Ordenamiento por paciente y fecha con filtrado manual de motivos. Debe verificarse si la versión modular oficial lo cubre completo.*

In [ ]:
# 1. ORDEN CRÍTICO: Por paciente y luego por fecha cronológica
df_base = df_base.sort_values(['paciente_id', 'fecha_ingreso']).reset_index(drop=True)

# 2. AGRUPACIÓN: Sacamos la "foto completa" de cada paciente
df_trayectorias = df_base.groupby('paciente_id').agg(
    fecha_ingreso_primera=('fecha_ingreso', 'first'),
    fecha_egreso_ultima=('fecha_egreso', 'last'),
    motivo_ultimo_egreso=('motivo_egreso', 'last'), # El estado FINAL del expediente
    cantidad_nodos=('hospital_origen', 'count')
).reset_index()

# 3. CÁLCULOS DERIVADOS
df_trayectorias['cantidad_traslados'] = df_trayectorias['cantidad_nodos'] - 1

# Días en el sistema (usando tu método de redondeo hacia arriba)
df_trayectorias['dias_en_sistema'] = np.ceil(
    (df_trayectorias['fecha_egreso_ultima'] - df_trayectorias['fecha_ingreso_primera']).dt.total_seconds() / 86400
).astype('Int64')

In [ ]:
df_base.groupby('paciente_id')['motivo_egreso'].last().value_counts()

In [ ]:
# 4. DEFINIR CIERRES VÁLIDOS DE HISTORIA CLÍNICA
# Usamos exactamente los strings que confirmamos en el value_counts
motivos_validos = [
    'alta-domiciliaria',         # Cierre natural
    'muerte',                    # Cierre natural
    'traslado-otro',             # Salió de nuestra red (cierre válido para nuestro alcance)
    'traslado-extra-sanitario'   # Traslado a hotel/domicilio (cierre natural)
]

# 5. APLICAR FILTRO DE CALIDAD
# Filtramos pacientes cuyo ÚLTIMO estado sea un cierre válido
mask_cierre_valido = df_trayectorias['motivo_ultimo_egreso'].isin(motivos_validos)

# También nos aseguramos de que no haya días negativos por errores de fechas
mask_dias_validos = (df_trayectorias['dias_en_sistema'] >= 0) & (df_trayectorias['dias_en_sistema'].notna())

# Creamos el DataFrame ultra-limpio
df_limpio = df_trayectorias[mask_cierre_valido & mask_dias_validos].copy()

# 6. PRINT DE AUDITORÍA Y CALIDAD DE DATOS
print("--- REPORTE DE LIMPIEZA DE TRAYECTORIAS ---")
print(f"Total pacientes originales (únicos): {len(df_trayectorias)}")
print(f"Pacientes retenidos (trayectoria completa y cerrada): {len(df_limpio)}")
print(f"Pacientes descartados (truncados/anulados/otro): {len(df_trayectorias) - len(df_limpio)}")

In [ ]:
# Separar los universos limpios
df_sin_traslado = df_limpio[df_limpio['cantidad_traslados'] == 0]
df_con_traslado = df_limpio[df_limpio['cantidad_traslados'] > 0]

COLOR_PRINCIPAL = '#4a7abc'  # Global
COLOR_NEUTRO = '#95a5a6'     # Sin Traslado
COLOR_ACENTO = '#5cb85c'     # Trasladados

max_plot = int(df_limpio['dias_en_sistema'].quantile(0.99)) # Cortamos el 1% de outliers extremos
bins_5dias = np.arange(0, max_plot + 5, 5)

config_graficos = [
    (df_limpio, "A. Pacientes Totales (Limpios)", COLOR_PRINCIPAL),
    (df_sin_traslado, "B. Sin Traslados", COLOR_NEUTRO),
    (df_con_traslado, "C. Con Traslados", COLOR_ACENTO)
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
fig.patch.set_facecolor('white')

for i, (ax, (df_plot, titulo, color)) in enumerate(zip(axes, config_graficos)):
    sns.histplot(data=df_plot, x='dias_en_sistema', bins=bins_5dias, 
                 color=color, stat='count', alpha=0.7, ax=ax, edgecolor='white')
    
    mediana = df_plot['dias_en_sistema'].median()
    ax.axvline(mediana, color='#e67e22', linestyle='--', linewidth=2.5, label=f'Mediana: {mediana:.1f} días')
    
    ax.set_xlim(0, max_plot)
    ax.set_xlabel("") 
    if i == 0:
        ax.set_ylabel("Cantidad de pacientes", fontsize=16, fontweight='bold')
    
    ax.set_title(titulo, fontweight='bold')
    ax.legend(frameon=False, loc='upper right')

fig.text(0.5, 0.02, 'Días en el sistema (Estadía Total Real)', ha='center', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1]) 
plt.show()